In [52]:
!pip install requests
!pip install feedparser
!pip install feedparser
!pip install feedparser beautifulsoup4 requests
!pip install pandas requests feedparser beautifulsoup4 trafilatura lxml

In [63]:
# Install once in Colab if needed:
# !pip install pandas requests feedparser beautifulsoup4 trafilatura lxml

import logging
import pandas as pd
import requests
import feedparser
import trafilatura
from bs4 import BeautifulSoup
from urllib.parse import quote

# -----------------------------------
# HIDE TRAFILATURA WARNING NOISE
# -----------------------------------
logging.getLogger("trafilatura").setLevel(logging.ERROR)
logging.getLogger("trafilatura.core").setLevel(logging.ERROR)
logging.getLogger("trafilatura.utils").setLevel(logging.ERROR)

# -----------------------------------
# COLOR / SYMBOL HELPERS
# -----------------------------------
RESET = "\033[0m"
GREEN = "\033[92m"
RED = "\033[91m"
BLUE = "\033[94m"
CYAN = "\033[96m"
BOLD = "\033[1m"

def color_text(text, color):
    return f"{color}{text}{RESET}"

def good_text(text):
    return color_text(text, GREEN)

def bad_text(text):
    return color_text(text, RED)

def neutral_text(text):
    return color_text(text, BLUE)

def header_text(text):
    return color_text(text, CYAN + BOLD)

def trend_symbol(trend_classification, setup_type):
    if trend_classification == "Bull market-like condition" or setup_type in ["Support Bounce", "Breakout Forming", "Bullish Pullback"]:
        return "🟢"
    elif trend_classification == "Bear market-like condition" or setup_type in ["Weak Near Support", "Bearish / Weak Trend"]:
        return "🔴"
    else:
        return "🟡"

def format_signed_pct(value):
    return f"{value:+.2f}%"

def price_vs_level_text(label, value, reference, higher_is_good=True):
    if pd.isna(value):
        return neutral_text(f"{label}: N/A")

    if higher_is_good:
        if value >= reference:
            return good_text(f"{label}: {round(value, 2)}")
        return bad_text(f"{label}: {round(value, 2)}")
    else:
        if value <= reference:
            return good_text(f"{label}: {round(value, 2)}")
        return bad_text(f"{label}: {round(value, 2)}")

def value_text(label, value, status="neutral", suffix=""):
    text = f"{label}: {value}{suffix}"
    if status == "good":
        return good_text(text)
    elif status == "bad":
        return bad_text(text)
    else:
        return neutral_text(text)

# -----------------------------------
# PEER / SIMILAR TICKER MAP
# -----------------------------------
PEER_MAP = {
    "AAPL": ["MSFT", "GOOGL", "AMZN", "META", "NVDA", "QQQ"],
    "MSFT": ["AAPL", "GOOGL", "AMZN", "META", "NVDA", "QQQ"],
    "NVDA": ["AMD", "TSM", "AVGO", "QCOM", "SOXX", "SMH"],
    "AMD": ["NVDA", "INTC", "QCOM", "AVGO", "SOXX", "SMH"],
    "TSLA": ["RIVN", "LCID", "NIO", "GM", "F", "XLY"],
    "META": ["GOOGL", "SNAP", "PINS", "RDDT", "QQQ", "XLC"],
    "AMZN": ["WMT", "COST", "SHOP", "EBAY", "XLY", "QQQ"],
    "GOOGL": ["META", "MSFT", "AAPL", "QQQ", "XLC", "SNAP"],
    "QQQ": ["SPY", "IWM", "SMH", "SOXX", "AAPL", "MSFT"],
    "SPY": ["QQQ", "IWM", "DIA", "VOO", "IVV"],
    "IWM": ["SPY", "QQQ", "DIA"],
    "SOXL": ["SOXX", "SMH", "NVDA", "AMD", "AVGO", "TSM"],
    "SOXX": ["SMH", "NVDA", "AMD", "AVGO", "TSM", "QCOM"],
    "SMH": ["SOXX", "NVDA", "AMD", "AVGO", "TSM", "QCOM"],
    "AVGO": ["NVDA", "AMD", "QCOM", "TSM", "SOXX", "SMH"],
    "TSM": ["NVDA", "AMD", "AVGO", "QCOM", "SOXX", "SMH"],
    "QCOM": ["AVGO", "AMD", "NVDA", "TSM", "SOXX", "SMH"],
    "RIVN": ["TSLA", "LCID", "NIO", "GM", "F"],
    "LCID": ["TSLA", "RIVN", "NIO"],
    "NIO": ["TSLA", "RIVN", "LCID"],
    "PLTR": ["SNOW", "CRM", "MDB", "AI", "QQQ"],
    "SNOW": ["MDB", "CRM", "PLTR", "DDOG", "QQQ"],
    "CRM": ["NOW", "MDB", "SNOW", "PLTR", "QQQ"],
}

DEFAULT_PEERS = ["SPY", "QQQ", "IWM", "AAPL", "MSFT", "NVDA"]

# -----------------------------------
# NEWS SENTIMENT HELPERS
# Used only in background
# -----------------------------------
def estimate_article_sentiment(text):
    positive_words = [
        "beat", "beats", "growth", "strong", "surge", "surges", "profit", "profits",
        "record", "upgrade", "upgrades", "gain", "gains", "positive", "expansion",
        "bullish", "outperform", "recovery", "improved", "improves", "partnership",
        "launch", "launched", "success", "successful", "higher", "rise", "rises",
        "raised guidance", "approval", "approves", "contract win", "buyback",
        "acquisition", "demand", "rebound", "momentum"
    ]

    negative_words = [
        "miss", "misses", "loss", "losses", "weak", "drop", "drops", "decline",
        "declines", "lawsuit", "probe", "investigation", "downgrade", "downgrades",
        "fraud", "recall", "delay", "delays", "cut", "cuts", "negative", "bearish",
        "fall", "falls", "lower", "warning", "risk", "risks", "debt",
        "cuts guidance", "regulatory", "tariff", "tariffs", "selloff", "offering",
        "dilution", "missed estimates", "layoffs"
    ]

    text_lower = str(text).lower()
    positive_score = sum(text_lower.count(word) for word in positive_words)
    negative_score = sum(text_lower.count(word) for word in negative_words)

    return positive_score - negative_score


def extract_article_text(url, rss_summary=""):
    headers = {"User-Agent": "Mozilla/5.0"}
    article_text = ""

    try:
        downloaded = trafilatura.fetch_url(url)
        if downloaded:
            extracted = trafilatura.extract(
                downloaded,
                url=url,
                include_comments=False,
                include_tables=False
            )
            if extracted and len(extracted.strip()) > 400:
                article_text = extracted.strip()
    except Exception:
        pass

    if len(article_text) < 400:
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")

            for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
                tag.extract()

            article_candidates = soup.select("article, main, [role='main']")
            best_text = ""

            for block in article_candidates:
                paragraphs = block.find_all("p")
                combined = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
                if len(combined) > len(best_text):
                    best_text = combined

            if len(best_text) > 300:
                article_text = best_text
            else:
                paragraphs = soup.find_all("p")
                article_text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)

        except Exception:
            pass

    if len(article_text.strip()) < 150 and rss_summary:
        article_text = rss_summary

    return article_text.strip()


def get_background_news_score(symbol):
    query = quote(f"{symbol} stock OR earnings OR guidance OR outlook")
    news_url = f"https://news.google.com/rss/search?q={query}"
    feed = feedparser.parse(news_url)

    total_score = 0

    if not getattr(feed, "entries", None):
        return 0

    for entry in feed.entries[:5]:
        title = entry.get("title", "")
        link = entry.get("link", "")
        summary = entry.get("summary", "")

        try:
            article_text = extract_article_text(link, summary)
            combined_text = f"{title}. {summary}. {article_text}"
        except Exception:
            combined_text = f"{title}. {summary}"

        total_score += estimate_article_sentiment(combined_text)

    return total_score


# -----------------------------------
# TECHNICAL INDICATORS
# -----------------------------------
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def is_high_volatility_symbol(symbol):
    leveraged_symbols = {
        "SOXL", "TQQQ", "SQQQ", "SPXL", "SPXS",
        "LABU", "LABD", "TECL", "FNGU", "FNGD",
        "NVDL", "NVDX", "TSLL", "TSLQ", "BULZ", "BERZ"
    }
    return symbol.upper() in leveraged_symbols


def classify_setup(close_price, ema20, ema50, rsi, volume_ratio, news_score, breakout_20d, support_20d):
    if pd.isna(rsi):
        return "Not enough data"

    support_distance_pct = ((close_price - support_20d) / support_20d) * 100 if support_20d != 0 else 999
    breakout_distance_pct = ((breakout_20d - close_price) / close_price) * 100 if close_price != 0 else 999

    # Near support
    if support_distance_pct <= 2.5:
        if rsi >= 45 and news_score >= 0:
            return "Support Bounce"
        elif rsi < 45:
            return "Weak Near Support"
        else:
            return "Neutral"

    # Near breakout
    if breakout_distance_pct <= 2.5:
        if close_price > ema20 and volume_ratio >= 0.9 and news_score >= 0:
            return "Breakout Forming"
        else:
            return "Near Breakout"

    # Bullish pullback
    if close_price > ema20 and ema20 > ema50 and 50 <= rsi <= 68 and news_score >= 0:
        return "Bullish Pullback"

    # Bearish trend
    if close_price < ema20 and ema20 < ema50 and 32 <= rsi <= 50 and news_score <= 0:
        return "Bearish / Weak Trend"

    return "Neutral"


def get_trade_levels(symbol, close_price, support_20d, breakout_20d, ema20, atr_like, setup_type):
    atr_like = max(float(atr_like), close_price * 0.01)

    if is_high_volatility_symbol(symbol):
        short_rr = 0.8
        long_rr = 1.2
        max_short_gain_pct = 0.035
        max_long_gain_pct = 0.06
    else:
        short_rr = 1.0
        long_rr = 1.5
        max_short_gain_pct = 0.04
        max_long_gain_pct = 0.07

    # 1) Good support bounce
    if setup_type == "Support Bounce":
        buy_price = close_price
        stop_loss = support_20d - (0.5 * atr_like)
        risk_per_share = max(buy_price - stop_loss, close_price * 0.01)

        target1_by_risk = buy_price + (short_rr * risk_per_share)
        target2_by_risk = buy_price + (long_rr * risk_per_share)

        sell_target_1 = min(
            target1_by_risk,
            ema20 if ema20 > buy_price else buy_price * (1 + max_short_gain_pct)
        )
        sell_target_2 = min(
            target2_by_risk,
            buy_price * (1 + max_long_gain_pct)
        )

    # 2) Breakout setup
    elif setup_type in ["Breakout Forming", "Near Breakout"]:
        buy_price = breakout_20d * 1.002
        stop_loss = breakout_20d - (0.75 * atr_like)

        if stop_loss >= buy_price:
            stop_loss = buy_price - max(atr_like, close_price * 0.01)

        risk_per_share = max(buy_price - stop_loss, close_price * 0.01)

        target1_by_risk = buy_price + (short_rr * risk_per_share)
        target2_by_risk = buy_price + (long_rr * risk_per_share)

        sell_target_1 = min(target1_by_risk, buy_price * (1 + max_short_gain_pct))
        sell_target_2 = min(target2_by_risk, buy_price * (1 + max_long_gain_pct))

    # 3) Bullish pullback
    elif setup_type == "Bullish Pullback":
        buy_price = min(close_price, ema20 * 1.01)
        stop_loss = buy_price - atr_like
        risk_per_share = max(buy_price - stop_loss, close_price * 0.01)

        target1_by_risk = buy_price + (short_rr * risk_per_share)
        target2_by_risk = buy_price + (long_rr * risk_per_share)

        sell_target_1 = min(target1_by_risk, buy_price * (1 + max_short_gain_pct))
        sell_target_2 = min(target2_by_risk, buy_price * (1 + max_long_gain_pct))

    # 4) Weak / Neutral setup
    else:
        # If price is near support, use current price area, not a far trigger
        support_distance_pct = ((close_price - support_20d) / support_20d) * 100 if support_20d != 0 else 999
        breakout_distance_pct = ((breakout_20d - close_price) / close_price) * 100 if close_price != 0 else 999

        if support_distance_pct <= 3.0:
            # Near support: use current price as possible speculative entry
            buy_price = close_price
            stop_loss = support_20d - (0.5 * atr_like)
        elif breakout_distance_pct <= 3.0:
            # Near breakout: use breakout confirmation
            buy_price = breakout_20d * 1.002
            stop_loss = breakout_20d - (0.75 * atr_like)
        else:
            # Middle of range: use a modest pullback-style entry near current price
            buy_price = close_price
            stop_loss = close_price - atr_like

        if stop_loss >= buy_price:
            stop_loss = buy_price - max(atr_like, close_price * 0.01)

        risk_per_share = max(buy_price - stop_loss, close_price * 0.01)

        target1_by_risk = buy_price + (short_rr * risk_per_share)
        target2_by_risk = buy_price + (long_rr * risk_per_share)

        # Keep targets realistic for a short swing
        sell_target_1 = min(
            target1_by_risk,
            ema20 if ema20 > buy_price else buy_price * (1 + max_short_gain_pct)
        )
        sell_target_2 = min(
            target2_by_risk,
            buy_price * (1 + max_long_gain_pct)
        )

    return {
        "buy_price": buy_price,
        "stop_loss": stop_loss,
        "sell_target_1": sell_target_1,
        "sell_target_2": sell_target_2
    }


def fetch_price_data(symbol):
    stooq_symbol = symbol.lower() + ".us"
    stock_url = f"https://stooq.com/q/d/l/?s={stooq_symbol}&i=d"
    df = pd.read_csv(stock_url)

    if df.empty or "Close" not in df.columns:
        return None

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Open", "High", "Low", "Close"])

    if df.empty or len(df) < 20:
        return None

    return df


def analyze_symbol(symbol, use_news=True):
    df = fetch_price_data(symbol)
    if df is None:
        return None

    df["EMA20"] = df["Close"].ewm(span=20, adjust=False).mean()
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["RSI14"] = calculate_rsi(df["Close"], 14)
    df["AvgVol20"] = df["Volume"].rolling(20).mean()
    df["Range"] = df["High"] - df["Low"]

    latest_row = df.iloc[-1]

    close_price = latest_row["Close"]
    open_price = latest_row["Open"]
    high_price = latest_row["High"]
    low_price = latest_row["Low"]
    volume = latest_row["Volume"]
    company_date = latest_row["Date"].date()

    daily_change_percent = ((close_price - open_price) / open_price) * 100 if open_price != 0 else 0

    last_252_days = df.tail(252)
    high_52_week = last_252_days["High"].max()
    low_52_week = last_252_days["Low"].min()

    last_40_days = df.tail(40)
    recent_low = last_40_days["Low"].min()
    recent_high = last_40_days["High"].max()

    percent_from_recent_low = ((close_price - recent_low) / recent_low) * 100 if recent_low != 0 else 0
    percent_from_recent_high = ((close_price - recent_high) / recent_high) * 100 if recent_high != 0 else 0

    if percent_from_recent_low >= 20:
        trend_classification = "Bull market-like condition"
    elif percent_from_recent_high <= -20:
        trend_classification = "Bear market-like condition"
    else:
        trend_classification = "Neutral / no clear bull-bear condition"

    ema20 = latest_row["EMA20"]
    ema50 = latest_row["EMA50"]
    rsi14 = latest_row["RSI14"]
    avg_vol20 = latest_row["AvgVol20"]
    volume_ratio = (volume / avg_vol20) if pd.notna(avg_vol20) and avg_vol20 not in [0, None] else 0

    breakout_20d = df.tail(20)["High"].max()
    support_20d = df.tail(20)["Low"].min()
    atr_like = df.tail(14)["Range"].mean()

    news_score_total = get_background_news_score(symbol) if use_news else 0

    setup_type = classify_setup(
        close_price=close_price,
        ema20=ema20,
        ema50=ema50,
        rsi=rsi14,
        volume_ratio=volume_ratio,
        news_score=news_score_total,
        breakout_20d=breakout_20d,
        support_20d=support_20d
    )

    trade_levels = get_trade_levels(
        symbol=symbol,
        close_price=close_price,
        support_20d=support_20d,
        breakout_20d=breakout_20d,
        ema20=ema20,
        atr_like=atr_like,
        setup_type=setup_type
    )

    return {
        "symbol": symbol,
        "date": company_date,
        "open": open_price,
        "high": high_price,
        "low": low_price,
        "close": close_price,
        "daily_change_percent": daily_change_percent,
        "high_52_week": high_52_week,
        "low_52_week": low_52_week,
        "ema20": ema20,
        "ema50": ema50,
        "rsi14": rsi14,
        "volume": volume,
        "avg_vol20": avg_vol20,
        "volume_ratio": volume_ratio,
        "breakout_20d": breakout_20d,
        "support_20d": support_20d,
        "recent_low": recent_low,
        "recent_high": recent_high,
        "percent_from_recent_low": percent_from_recent_low,
        "percent_from_recent_high": percent_from_recent_high,
        "trend_classification": trend_classification,
        "setup_type": setup_type,
        "trade_levels": trade_levels,
    }


def get_peer_list(symbol):
    return PEER_MAP.get(symbol.upper(), DEFAULT_PEERS)


def find_similar_tickers(main_result, peer_symbols, top_n=5):
    rows = []

    for peer in peer_symbols:
        try:
            peer_result = analyze_symbol(peer, use_news=False)
            if peer_result is None:
                continue

            score = 0
            if peer_result["setup_type"] == main_result["setup_type"]:
                score += 4

            score += max(0, 3 - abs(peer_result["rsi14"] - main_result["rsi14"]) / 10)
            score += max(0, 2 - abs(peer_result["volume_ratio"] - main_result["volume_ratio"]))
            score += max(0, 2 - abs(peer_result["percent_from_recent_high"] - main_result["percent_from_recent_high"]) / 10)

            rows.append({
                "Ticker": peer_result["symbol"],
                "Setup": peer_result["setup_type"],
                "Close": round(peer_result["close"], 2),
                "RSI": round(peer_result["rsi14"], 2) if pd.notna(peer_result["rsi14"]) else None,
                "Volume Ratio": round(peer_result["volume_ratio"], 2),
                "Buy Price": round(peer_result["trade_levels"]["buy_price"], 2),
                "Stop Loss": round(peer_result["trade_levels"]["stop_loss"], 2),
                "Target 1": round(peer_result["trade_levels"]["sell_target_1"], 2),
                "Target 2": round(peer_result["trade_levels"]["sell_target_2"], 2),
                "Similarity Score": round(score, 2)
            })
        except Exception:
            continue

    if not rows:
        return pd.DataFrame()

    similar_df = pd.DataFrame(rows)
    similar_df = similar_df.sort_values(by=["Similarity Score", "Ticker"], ascending=[False, True]).head(top_n)
    return similar_df


# -----------------------------------
# MAIN
# -----------------------------------
symbol = input("Enter stock ticker: ").strip().upper()

try:
    result = analyze_symbol(symbol, use_news=True)

    if result is None:
        print("Stock not found or no data available.")
    else:
        symbol_icon = trend_symbol(result["trend_classification"], result["setup_type"])

        print("\n" + header_text("Stock Information"))
        print(header_text("-----------------"))
        print(header_text(f"Ticker: {result['symbol']} {symbol_icon}"))

        print(neutral_text(f"Date: {result['date']}"))
        print(neutral_text(f"Open: {round(result['open'], 2)}"))
        print(neutral_text(f"High: {round(result['high'], 2)}"))
        print(neutral_text(f"Low: {round(result['low'], 2)}"))
        print(neutral_text(f"Close: {round(result['close'], 2)}"))

        if result["daily_change_percent"] > 0:
            print(good_text(f"Daily Change: {format_signed_pct(result['daily_change_percent'])}"))
        elif result["daily_change_percent"] < 0:
            print(bad_text(f"Daily Change: {format_signed_pct(result['daily_change_percent'])}"))
        else:
            print(neutral_text(f"Daily Change: {format_signed_pct(result['daily_change_percent'])}"))

        print(neutral_text(f"52-Week High: {round(result['high_52_week'], 2)}"))
        print(neutral_text(f"52-Week Low: {round(result['low_52_week'], 2)}"))

        if result["close"] > result["ema20"]:
            print(good_text(f"20 EMA: {round(result['ema20'], 2)}"))
        else:
            print(bad_text(f"20 EMA: {round(result['ema20'], 2)}"))

        if result["ema20"] > result["ema50"]:
            print(good_text(f"50 EMA: {round(result['ema50'], 2)}"))
        else:
            print(bad_text(f"50 EMA: {round(result['ema50'], 2)}"))

        rsi = result["rsi14"]
        if pd.notna(rsi):
            if 45 <= rsi <= 68:
                print(good_text(f"RSI (14): {round(rsi, 2)}"))
            elif rsi < 35 or rsi > 75:
                print(bad_text(f"RSI (14): {round(rsi, 2)}"))
            else:
                print(neutral_text(f"RSI (14): {round(rsi, 2)}"))
        else:
            print(neutral_text("RSI (14): N/A"))

        print(neutral_text(f"Volume: {int(result['volume']) if pd.notna(result['volume']) else 'N/A'}"))

        if pd.notna(result["avg_vol20"]):
            print(neutral_text(f"20-Day Avg Volume: {round(result['avg_vol20'], 0)}"))
        else:
            print(neutral_text("20-Day Avg Volume: N/A"))

        if result["volume_ratio"] >= 1.0:
            print(good_text(f"Volume Ratio: {round(result['volume_ratio'], 2)}"))
        elif result["volume_ratio"] < 0.8:
            print(bad_text(f"Volume Ratio: {round(result['volume_ratio'], 2)}"))
        else:
            print(neutral_text(f"Volume Ratio: {round(result['volume_ratio'], 2)}"))

        print(neutral_text(f"20-Day Breakout Level: {round(result['breakout_20d'], 2)}"))
        print(neutral_text(f"20-Day Support Level: {round(result['support_20d'], 2)}"))
        print(neutral_text(f"Recent 2-Month Low: {round(result['recent_low'], 2)}"))
        print(neutral_text(f"Recent 2-Month High: {round(result['recent_high'], 2)}"))

        if result["percent_from_recent_low"] >= 10:
            print(good_text(f"% From Recent Low: {round(result['percent_from_recent_low'], 2)} %"))
        elif result["percent_from_recent_low"] <= 3:
            print(bad_text(f"% From Recent Low: {round(result['percent_from_recent_low'], 2)} %"))
        else:
            print(neutral_text(f"% From Recent Low: {round(result['percent_from_recent_low'], 2)} %"))

        if result["percent_from_recent_high"] >= -5:
            print(good_text(f"% From Recent High: {round(result['percent_from_recent_high'], 2)} %"))
        elif result["percent_from_recent_high"] <= -15:
            print(bad_text(f"% From Recent High: {round(result['percent_from_recent_high'], 2)} %"))
        else:
            print(neutral_text(f"% From Recent High: {round(result['percent_from_recent_high'], 2)} %"))

        if result["trend_classification"] == "Bull market-like condition":
            print(good_text(f"Trend Classification: {result['trend_classification']}"))
        elif result["trend_classification"] == "Bear market-like condition":
            print(bad_text(f"Trend Classification: {result['trend_classification']}"))
        else:
            print(neutral_text(f"Trend Classification: {result['trend_classification']}"))

        if result["setup_type"] in ["Support Bounce", "Breakout Forming", "Bullish Pullback"]:
            print(good_text(f"Swing Setup: {result['setup_type']}"))
        elif result["setup_type"] in ["Weak Near Support", "Bearish / Weak Trend"]:
            print(bad_text(f"Swing Setup: {result['setup_type']}"))
        else:
            print(neutral_text(f"Swing Setup: {result['setup_type']}"))

        print("\n" + header_text("Swing Trade Levels"))
        print(header_text("------------------"))
        print(neutral_text(f"Buy Price: {round(result['trade_levels']['buy_price'], 2)}"))
        print(bad_text(f"Stop Loss: {round(result['trade_levels']['stop_loss'], 2)}"))
        print(good_text(f"Sell Target 1 (Short Term): {round(result['trade_levels']['sell_target_1'], 2)}"))
        print(good_text(f"Sell Target 2 (Long Term): {round(result['trade_levels']['sell_target_2'], 2)}"))

        peer_symbols = [p for p in get_peer_list(symbol) if p != symbol]
        similar_df = find_similar_tickers(result, peer_symbols, top_n=5)

        print("\n" + header_text("Similar Ticker Setups To Look Up"))
        print(header_text("--------------------------------"))
        if similar_df.empty:
            print(neutral_text("No similar ticker setups found."))
        else:
            print(similar_df.to_string(index=False))

except Exception as e:
    print("Error fetching stock data or processing analysis.")
    print("Details:", e)

Enter stock ticker: SOXL

Stock Information
-----------------
Ticker: SOXL 🔴
Date: 2026-03-06
Open: 50.09
High: 53.36
Low: 46.8
Close: 47.89
Daily Change: -4.39%
52-Week High: 72.36
52-Week Low: 7.22
20 EMA: 60.41
50 EMA: 57.72
RSI (14): 30.46
Volume: 125241492
20-Day Avg Volume: 83643603.0
Volume Ratio: 1.5
20-Day Breakout Level: 72.36
20-Day Support Level: 46.8
Recent 2-Month Low: 46.8
Recent 2-Month High: 72.36
% From Recent Low: 2.33 %
% From Recent High: -33.82 %
Trend Classification: Bear market-like condition
Swing Setup: Weak Near Support

Swing Trade Levels
------------------
Buy Price: 47.89
Stop Loss: 44.44
Sell Target 1 (Short Term): 50.65
Sell Target 2 (Long Term): 50.76

Similar Ticker Setups To Look Up
--------------------------------
Ticker             Setup  Close   RSI  Volume Ratio  Buy Price  Stop Loss  Target 1  Target 2  Similarity Score
   AMD Weak Near Support 192.44 40.45          0.91     192.44     184.30    200.58    204.65              8.82
  SOXX Weak Near